In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/processed/movies_metadata.csv")
print(df.shape)
print(df.columns.tolist())


(45466, 6)
['id', 'release_date', 'budget', 'revenue', 'runtime', 'popularity']


In [2]:
#Dimensions 
print("Shape (columns, rows):", df.shape)

#Data Types 
print("\nData types:")
print (df.dtypes)

Shape (columns, rows): (45466, 6)

Data types:
id              float64
release_date        str
budget              str
revenue         float64
runtime         float64
popularity      float64
dtype: object


In [3]:
#Converting ID to integer 
df['id'] = df['id'].astype('Int64').astype(str)

#Checking for any missing ID values 
df['id'].isna().sum()
df[df['id'].isna()]

#Dropping movies with junk data 
df = df.dropna(subset=['id']).reset_index(drop=True)
print(df.shape)

#Checking for ID duplicates
print(df['id'].is_unique)
print(df['id'].duplicated().sum())

#Inspecting dupes 
dupes = df[df['id'].duplicated(keep=False)].sort_values('id')
print(len(dupes))
print (dupes['id'].nunique())

print(df.duplicated().sum())

#Removing exact copies and show conflicting pairs 
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)
print(df['id'].duplicated().sum())

dupes = df[df['id'].duplicated(keep=False)].sort_values('id')

#Confirming that 'popularity' is the only difference 
cols = ['release_date', 'budget', 'revenue', 'runtime', 'popularity']
differs = dupes.groupby('id')[cols].nunique(dropna=False).gt(1)
print(differs.sum())

#inspecting gaps
dupes.sort_values(['id', 'popularity'])

#Keep the highest score of popularity within the remaining 13 pairs
df = (df.sort_values(['id', 'popularity'], ascending=[True, False])
      .drop_duplicates(subset='id', keep='first')
      .reset_index(drop=True))

#Confirming there is only one row per id
assert df['id'].is_unique
print(df.shape)

(45463, 6)
False
30
59
29
17
(45446, 6)
13
release_date     0
budget           0
revenue          0
runtime          0
popularity      13
dtype: int64
(45433, 6)


In [4]:
#Loading raw columns 
cols = ['id', 'original_language', 'belongs_to_collection', 'genres',
        'production_companies', 'production_countries',
        'title', 'original_title', 'overview', 'status']
raw = pd.read_csv('../../data/raw/movies_metadata.csv', usecols=cols, low_memory=False)
print(raw.shape)

(45466, 10)


In [5]:
#Cleaning IDs and checking for duplicates 
raw['id'] = pd.to_numeric(raw['id'], errors='coerce')
raw = raw.dropna(subset=['id'])
raw['id'] = raw['id'].astype('Int64').astype(str)
print(raw.shape)
print(raw['id'].duplicated().sum())

#Removing duplicate ids and merge 
raw = raw.drop_duplicates(subset='id')

before = df.shape
df = df.merge(raw, on='id', how='left')
print(before, '->', df.shape)

(45463, 10)
30
(45433, 6) -> (45433, 15)


In [6]:
#Dropping rows with no original_language
df = df.dropna(subset=['original_language']).reset_index(drop=True)
print(df.shape)                                 
print(df['original_language'].isna().sum()) 

(45422, 15)
0


In [7]:
#Inspect language frequencies 
df['original_language'].value_counts

#Expanding to to see the whole list
counts = df['original_language'].value_counts()
pd.set_option('display.max_rows', None)

#Apply cutoff to languages with <100 movies
MIN_COUNT = 100
common = counts[counts >= MIN_COUNT].index
df['target_original_language'] = df['original_language'].where(
    df['original_language'].isin(common),'Other'
)

print(df['target_original_language'].value_counts())
print(df['target_original_language'].nunique())

target_original_language
en       32251
fr        2436
it        1529
ja        1347
de        1079
es         994
Other      902
ru         826
hi         508
ko         444
zh         409
sv         383
pt         316
cn         313
fi         295
nl         248
da         224
pl         219
tr         150
cs         130
el         113
no         106
hu         100
fa         100
Name: count, dtype: int64
24


In [8]:
#Merging credits
credits = pd.read_csv('../../data/processed/credits.csv',
                      usecols=['id', 'director', 'top_five_cast', 'cast_size', 'crew_size'])
credits['id'] = pd.to_numeric(credits['id'], errors='coerce')
credits = credits.dropna(subset=['id'])
credits['id'] = credits['id'].astype('Int64').astype(str)
credits = credits.drop_duplicates(subset='id')

before = df.shape
df = df.merge(credits, on='id', how='left')
print("Credits merge: ", before, '->', df.shape)

df = df.rename(columns={'top_five_cast': 'top_cast'})
df['has_director'] = df['director'].notna() & (df['director'].astype(str).str.strip() != '')

#Merge keywords
kw = pd.read_csv('../../data/processed/keywords_clean.csv')
kw['id'] = pd.to_numeric(kw['id'], errors='coerce')
kw = kw.dropna(subset=['id'])
kw['id'] = kw['id'].astype('Int64').astype(str)
kw = kw.drop_duplicates(subset='id')

before = df.shape
df = df.merge(kw, on='id', how='left')
print("Keywords merge: ", before, '->', df.shape)

#Renaming keyword_names_clean
df = df.rename(columns={'keyword_names_clean': 'keywords'})


Credits merge:  (45422, 16) -> (45422, 20)
Keywords merge:  (45422, 21) -> (45422, 22)


In [9]:
#Converting budget and release_date 
df['budget'] = pd.to_numeric(df['budget'], errors='coerce')
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

#Date features
df['release_year'] = df['release_date'].dt.year
df['release_month'] = df['release_date'].dt.month
df['release_decade'] = (df['release_year'] // 10) * 10

In [10]:
#Log Features 
df['log_budget'] = np.log1p(df['budget'])
df['log_revenue'] = np.log1p(df['revenue'])
df['log_popularity'] = np.log1p(df['popularity'])

#Build the count and text features 
import ast
def count_items(value):
    try:
        return len(ast.literal_eval(value))
    except (ValueError, SyntaxError, TypeError):
        return 0

df['genre_count'] = df['genres'].apply(count_items)
df['company_count'] = df['production_companies'].apply(count_items)
df['country_count'] = df['production_countries'].apply(count_items)
df['keyword_count'] = df['keywords'].apply(count_items)  
df['has_collection'] = df['belongs_to_collection'].notna()
df['title_word_count'] = df['title'].fillna('').str.split().str.len()
df['overview_word_count'] = df['overview'].fillna('').str.split().str.len()
df['title_differs_from_original'] = df['title'] != df['original_title']

#Checking
print(df.shape)
assert df['id'].is_unique
df.head()

required = ['id', 'target_original_language',
            'release_year', 'release_month', 'release_decade',
            'log_budget', 'log_revenue', 'runtime', 'log_popularity',
            'genre_count', 'has_collection', 'company_count', 'country_count',
            'cast_size', 'crew_size', 'has_director',
            'keyword_count', 'title_word_count', 'overview_word_count', 'title_differs_from_original',
            'genres', 'production_companies', 'production_countries',
            'director', 'top_cast', 'keywords', 'title', 'original_title', 'overview', 'status']

print('missing:', [c for c in required if c not in df.columns])

(45422, 36)
missing: []


In [11]:
final = df[required].copy()
print(final.shape)
assert final['id'].is_unique

(45422, 30)


In [12]:
#Creating csv
final.to_csv('../../data/features/15_language_features_jessica.csv', index=False)

check = pd.read_csv('../../data/features/15_language_features_jessica.csv')
print(check.shape)                           # expect (45422, 30)
print(check['id'].is_unique)                 # expect True
print(check.columns.tolist() == required)    # expect True

(45422, 30)
True
True
